## Using STED-FM model in PhenoMe pipeline

### 0. Setup

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from stedfm import get_pretrained_model_v2
from phenome.utils import ModelWrapper
from phenome import PhenoMe, load_dinov2_model

# Initialize PhenoMe object
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pheno = PhenoMe(device=device, seed=42)

#### Choose dataset and model to use

**NAS:** 224x224 crops of STED images of PSD-95 under 4 experimental conditions: 0MgGlyBic, 48hTTX, Block, and GluGly.

**Optim:** 224x224 crops of STED images of 4 different proteins: CaMKII, PSD-95, Actin and Tubulin.

**Zooniverse:** Crops of individual synaptic protein clusters. This is the only dataset with masks appropriate for extracting properties.


In [ ]:
# Choose one of 3 datasets:
#DATASET_NAME = "NAS"  # PSD95 only
#DATASET_NAME = "Optim"
#DATASET_NAME = "Zooniverse"

# Choose one of 2 models from ("STED-FM", "DinoV2")
MODEL = "STED-FM"
#MODEL = "DinoV2"

#### Download and transform data

`DATASETS_PATH` can be changed to wherever the user wants the datasets to be downloaded to.

In [3]:
# Decide where to save datasets
DATASETS_PATH = "stedfm_datasets"
os.makedirs(DATASETS_PATH, exist_ok=True)

DATA_DIR = os.path.join(DATASETS_PATH, f"{DATASET_NAME}_dataset")
os.makedirs(DATA_DIR, exist_ok=True)

In [4]:
from stedfm_data import download_dataset, transform_dataset

# Download and transform dataset to fit PhenoMe's required structure
download_dataset(DATASET_NAME, DATA_DIR)
transform_dataset(DATASET_NAME, DATA_DIR, DATA_DIR)

Actin dataset already exists!
Done!
Transforming Actin dataset...


Saving tiles: 100%|██████████| 316/316 [00:00<00:00, 1636.11it/s]


Saved 316 tiles to stedfm_datasets/Actin_dataset/images and stedfm_datasets/Actin_dataset/masks
Metadata written to stedfm_datasets/Actin_dataset/metadata.csv
Done!


### 1. Wrap the STED-FM pretrained model

In [5]:
class STEDFMWrapper(ModelWrapper):
    def __init__(self, device):
        # Load the pretrained STED-FM model
        model, cfg = get_pretrained_model_v2(
            name = "mae-lightning-small",
            weights = "MAE_SMALL_STED", 
            in_channels=1,
            as_classifier=True,
        )

        # Initialize the parent class which moves the model to the device and sets it to eval()
        super().__init__(model=model, device=device)

    def _get_embeddings(self, tensor: torch.Tensor) -> torch.Tensor:
        out = self.model.forward_features(tensor)
        return out

if MODEL == "STED-FM":
    wrapper = STEDFMWrapper(device=device)
elif MODEL == "DinoV2":
    wrapper = load_dinov2_model(model_name="dinov2_vits14_reg", device=device)

in_channels 1
--- mae-lightning-small | (https://s3.valeria.science/flclab-foundation-models/models/mae-small-sted.zip) Downloading from URL and extracting zip ---

Loading state_dict from /Users/renaud/.stedfm/baselines/mae-small_STED/pl_checkpoint-999.pth
--- Loaded model mae-lightning-small with weights MAE_SMALL_STED ---
--- Added linear probe to all frozen blocks ---


### 2. Use custom transforms

The default PhenoMe transforms apply standardisation based on ImageNet values, which isn't appropriate for the STED-FM model.

In [6]:
import torchvision.transforms as T

TARGET_SIZE = 224

if MODEL == "STED-FM":
    # Transforms for STED-FM model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
    ])
elif MODEL == "DinoV2":
    # Transforms for DinoV2 model
    custom_transforms = T.Compose([
        T.ToTensor(),
        T.Resize((TARGET_SIZE, TARGET_SIZE), interpolation=T.InterpolationMode.NEAREST),
        T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225))
    ])

### 3. Load data (find_files)

We assume CSV-based metadata. The STED datasets were already normalized. If raw data is used, it should be normalized based on percentile values from **entire images**, not by crop.

In [7]:

from phenome import make_dataframe_metadata_fn
import pandas as pd

images_dir = os.path.join(DATA_DIR, "images")
masks_dir = os.path.join(DATA_DIR, "masks") # Optional
metadata_dir = os.path.join(DATA_DIR, "metadata.csv")

metadata_df = pd.read_csv(metadata_dir)
metadata_fn = make_dataframe_metadata_fn(metadata_df)
file_df = pheno.find_files(images_dir, mask_dir=masks_dir, metadata_fn=metadata_fn)

print(f"Found {len(file_df)} images")

Found 316 files total.
Matched masks: 316/316 images (0 missing).
DataFrame: 316 files, 10 columns.


Found 316 images


### 4. Process images

In [8]:
# Base path to outputs: checkpoints and plots
OUTPUT_PATH = "output"
os.makedirs(OUTPUT_PATH, exist_ok=True)

CHECKPOINTS_PATH = os.path.join(OUTPUT_PATH, "checkpoints")
os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [9]:
checkpoint_path = os.path.join(CHECKPOINTS_PATH, DATASET_NAME + f"_{MODEL}" + "_checkpoint.hdf5")

if MODEL == "STED-FM":
    pheno.process_images(wrapper, 
                        force_rgb=False,  # STED-FM model expects a single channel
                        checkpoint_path=checkpoint_path,
                        custom_transformations=custom_transforms,
                        )
elif MODEL == "DinoV2":
    pheno.process_images(wrapper, 
                    force_rgb=True,
                    channel_mode='combined',
                    checkpoint_path=checkpoint_path,
                    custom_transformations=custom_transforms,
                    )
pheno.compute_properties(property_preset="complete", checkpoint_path=checkpoint_path)

Processing batches:   0%|          | 0/10 [00:00<?, ?it/s]

Pipeline ready: 316 images available (lazy loading from output/checkpoints/Actin_STED-FM_checkpoint.hdf5).
Masks: 316/316 images.


Computing properties:   0%|          | 0/316 [00:00<?, ?img/s]

Checkpoint: saved 316 properties (refreshed).
NaNs detected in property columns (NaN count per column): {'area': 225, 'circularity': 225, 'eccentricity': 225, 'equivalent_diameter': 225, 'euler_number': 225, 'extent': 225, 'intensity_max_masked': 225, 'intensity_mean_masked': 225, 'intensity_min_masked': 225, 'intensity_std_masked': 225, 'major_axis_length': 225, 'minor_axis_length': 225, 'orientation': 225, 'perimeter': 225, 'ring_1_mean': 227, 'ring_1_std': 227, 'ring_2_mean': 225, 'ring_2_std': 225, 'ring_3_mean': 225, 'ring_3_std': 225, 'solidity': 225, 'texture_contrast': 225, 'texture_correlation': 225, 'texture_dissimilarity': 225, 'texture_energy': 225, 'texture_homogeneity': 225}. 227/316 images have at least one such NaN. Use detect_nan_properties() for per-column row indices.


,image_index,image_path,image_name,ring_1_mean,circularity,solidity,euler_number,intensity_max,perimeter,intensity_entropy,...,sharpness_metric,texture_energy,equivalent_diameter,texture_contrast,major_axis_length,ring_2_std,ring_3_std,intensity_max_masked,texture_homogeneity,intensity_std
0,0,stedfm_datasets/Actin_dataset/images/img_0.tif,img_0.tif,0.059125,0.468312,0.628231,2.0,0.549020,92.08326,4.262452,...,0.280081,0.44157,20.058506,68.475502,50.67165,0.031494,0.071051,0.329412,0.514776,0.036794
1,1,stedfm_datasets/Actin_dataset/images/img_1.tif,img_1.tif,NaN,NaN,NaN,NaN,0.152941,NaN,1.736438,...,0.212248,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.012553
2,2,stedfm_datasets/Actin_dataset/images/img_10.tif,img_10.tif,NaN,NaN,NaN,NaN,0.145098,NaN,2.080154,...,0.238423,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.010769
3,3,stedfm_datasets/Actin_dataset/images/img_100.tif,img_100.tif,NaN,NaN,NaN,NaN,0.356863,NaN,4.125574,...,0.289771,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.033500
4,4,stedfm_datasets/Actin_dataset/images/img_101.tif,img_101.tif,NaN,NaN,NaN,NaN,0.325490,NaN,3.500329,...,0.289099,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.034669
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
311,311,stedfm_datasets/Actin_dataset/images/img_95.tif,img_95.tif,NaN,NaN,NaN,NaN,0.006500,NaN,3.319693,...,0.420244,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000670
312,312,stedfm_datasets/Actin_dataset/images/img_96.tif,img_96.tif,NaN,NaN,NaN,NaN,0.145098,NaN,1.396533,...,0.176802,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.006179
313,313,stedfm_datasets/Actin_dataset/images/img_97.tif,img_97.tif,NaN,NaN,NaN,NaN,0.105882,NaN,1.170445,...,0.173261,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.005268
314,314,stedfm_datasets/Actin_dataset/images/img_98.tif,img_98.tif,NaN,NaN,NaN,NaN,0.074510,NaN,1.017694,...,0.139859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.003422


### 5. Explore results

In [10]:
# Launch interactive explorer (if in Jupyter)
explorer = pheno.create_interactive_explorer()

### Generate plots

In [ ]:
PLOTS_PATH = os.path.join(OUTPUT_PATH, "plots")
os.makedirs(PLOTS_PATH, exist_ok=True)

if DATASET_NAME == "NAS":
    color_by = "condition"
    fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg", exclude={"condition":"Block"})
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))
elif DATASET_NAME == "Optim":
    color_by = "label"
elif DATASET_NAME == "Zooniverse":
    color_by = "protein"
    fig=pheno.plot_tsne(color_by="synapse_class", return_fig=True, render_mode="svg")
    fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_noblock_scatterplot.svg"))

fig=pheno.plot_tsne(color_by=color_by, return_fig=True, render_mode="svg")
fig.write_image(os.path.join(PLOTS_PATH, f"{DATASET_NAME}_{MODEL}_scatterplot.svg"))